In [2]:
import sys 
sys.path.append( "../")
sys.path.append( "../../")

from datetime import datetime 

from typing import Dict, List, Tuple, Optional, Any 
from pathlib import Path 
from load_semantics import load_semantics, load_idiom_rules 
from semantics.semantic_models import * 
 
from typing import Any, Dict
import json
import yaml
from get_llm_model import *
import pandas as pd, numpy as np
import duckdb
import pprint 



imported


In [3]:
semantic_catalog, sql_idioms, semantic_context = load_semantics( Path("../semantics/") )
rules, idiom_context = load_idiom_rules( idiom = 'duckdb',path = Path("../semantics/idioms.json") )
 
pprint.pprint(semantic_catalog.tables[0].model_dump_json() )

named_table_models = { item.name: item for item in semantic_catalog.tables }


('{"name":"injectors","description":"Water-injection time series per injector '
 'well, subzone and '
 'sector","kind":"base","creation_date":null,"row_count":null,"columns":[{"name":"DATE","data_type":"timestamp","description":"Timestamp '
 'of injection '
 'observation."},{"name":"NAME","data_type":"string","description":"Injector '
 'well name. Unique identifier for the injector '
 'well"},{"name":"WATER_INJECTION_VOLUME","data_type":"float","description":"Injected '
 'water volume for the '
 'period."},{"name":"SUBZONE","data_type":"string","description":"Subzone '
 'name. A subzone indicates vertical '
 'interval"},{"name":"SECTOR","data_type":"integer","description":"Sector '
 'identifier. A sector indicates a geographical '
 'location"},{"name":"YEAR","data_type":"integer","description":"Year '
 'component of '
 'DATE."},{"name":"MONTH","data_type":"integer","description":"Month component '
 'of DATE."},{"name":"DAY","data_type":"integer","description":"Day component '
 'of '
 '

In [4]:
inj = pd.read_csv("../datasets/IX5I_4P/injectors.csv")
inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
inj['DAY']   = inj['DATE'].dt.day
inj['MONTH'] = inj['DATE'].dt.month
inj['YEAR']  = inj['DATE'].dt.year

pinj = pd.read_csv("../datasets/IX5I_4P/producers.csv")
pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
pinj['DAY']   = pinj['DATE'].dt.day
pinj['MONTH'] = pinj['DATE'].dt.month
pinj['YEAR']  = pinj['DATE'].dt.year


df_dict = {'injectors': inj, 'producers':pinj }

In [71]:


from typing import Iterable, Union


class Catalog:

    def __init__(self):
        self.tables: Dict[str, TableCard] = {}

    def clear(self):
        self.tables = {} 

    def initialize_from_named_dataframes( self, df_dict: Dict[str,pd.DataFrame], named_table_models ):
        
        self.clear() 

        for name,df in df_dict.items():
            model = named_table_models.get(name, None)
            if model:
                model.row_count = df.shape[0]

                dt = datetime.now() if hasattr(datetime, "now") else datetime.datetime.now()  
                model.creation_date = str( dt )
                
                self.tables[name] = model
            else:
                raise ValueError(f"Table named {name} is not in the known tables catalog")
       
    def register_table(self, table_card: TableCard ):
        dt = datetime.now() if hasattr(datetime, "now") else datetime.datetime.now()  
             
        table_card.creation_date = str( dt )
        self.tables[ table_card.name ] = table_card

    @staticmethod  
    def dataframe_to_table_card( df: pd.DataFrame, name, description, kind:Literal['base','derived'], **kwargs):
        dt = datetime.now() if hasattr(datetime, "now") else datetime.datetime.now()  
             
        cols = [ ColumnCard( name = col, data_type = str(df[col].dtype), description = None) for col in df.columns] 
        table_card = TableCard(
            name=name,
            description=description,
            kind = kind, 
            creation_date=str( dt ),
            row_count=df.shape[0],
            columns = cols,
            relationships = [] if not kwargs else kwargs.get('relationships', []),
            sql_examples  = [] if not kwargs else kwargs.get('sql_exampled',  [])

        )

        return table_card

    def __repr__(self) -> str:
        return self.snapshot()

    def snapshot(self, input_tables: None | TableCard | Iterable[TableCard] = None ) -> str: # pyright: ignore[reportArgumentType]
         
        
        tables = []
        
        items = (
            self.tables.values()
            if input_tables is None
            else [input_tables]
            if isinstance(input_tables, TableCard)
            else input_tables if isinstance(input_tables,Iterable)
            else list(input_tables)
        )
        for tc in  sorted( items,  key=lambda x: x.name):
            d = tc.model_dump(
                exclude_none=True,
                exclude_defaults=True,
                exclude_unset = True,
                mode = 'json',
                #exclude = {'columns'}
            )

            #cols = [ f"{c.name}: type:{c.data_type} " + f" description: {c.description}" if c.description else f"{c.name}: type:{c.data_type} " for c in tc.columns ]
            #d['columns'] = cols 
            # Optional pruning (VERY recommended)
            #d.pop("creation_date", None)

            tables.append(d)

        return yaml.dump(
            {"tables": tables},
            sort_keys=False,
            allow_unicode=True,
            width=1000  # avoid wrapping
        )#.replace('- name: ','\n- name:')

    

    def __getitem__(self, value):
        
        cards = None 
        if isinstance(value, slice):
            cards =  list(self.tables.values())[value] 

        if isinstance(value, str ):
            cards = self.tables[value]

        if isinstance(value, Iterable ):
            cards = [ self.tables[v] for v in value] 
 

        return cards  

catalog = Catalog()
catalog.initialize_from_named_dataframes( df_dict, named_table_models)

df = pd.DataFrame( {'name':['a','b'], 'age':[1,2]})
catalog.register_table( catalog.dataframe_to_table_card(df, name='aaa',description='sddgf', kind='derived'))
print(catalog.snapshot( catalog['aaa', 'injectors' ] ))


tables:
- name: aaa
  description: sddgf
  kind: derived
  creation_date: '2026-04-14 22:14:55.376319'
  row_count: 2
  columns:
  - name: name
    data_type: object
  - name: age
    data_type: int64
- name: injectors
  description: Water-injection time series per injector well, subzone and sector
  kind: base
  creation_date: '2026-04-14 22:14:55.375432'
  row_count: 490
  columns:
  - name: DATE
    data_type: timestamp
    description: Timestamp of injection observation.
  - name: NAME
    data_type: string
    description: Injector well name. Unique identifier for the injector well
  - name: WATER_INJECTION_VOLUME
    data_type: float
    description: Injected water volume for the period.
  - name: SUBZONE
    data_type: string
    description: Subzone name. A subzone indicates vertical interval
  - name: SECTOR
    data_type: integer
    description: Sector identifier. A sector indicates a geographical location
  - name: YEAR
    data_type: integer
    description: Year component

In [ ]:

class SmartData:

    def __init__(self):
        self._catalog = Catalog() 
        self.conn= duckdb.connect()

    def clear( self ):
        self._catalog.clear()
        self.conn.close()
        self.conn= duckdb.connect()

    def initialize_from_named_dataframes( self, df_dict: Dict[str,pd.DataFrame], named_table_models ):
        self.clear()

        try:
            conn = self.conn
            self._catalog.initialize_from_named_dataframes( df_dict, named_table_models )
            
            for name, df in df_dict.items():
                df = self.sanitize_df(df)
                conn.register(name, df)

        except Exception as e:
            print( str(e))
            self.clear()
            
    def catalog_snapshot(self, input_tables: None | str | Iterable[str] = None) -> str:
        if input_tables is None:
            return self._catalog.snapshot()
        if isinstance(input_tables, str):
            card = self._catalog.tables[input_tables]
            return self._catalog.snapshot(card)
        if isinstance(input_tables, Iterable):
            cards = [self._catalog.tables[name] for name in input_tables]
            return self._catalog.snapshot(cards)
        raise TypeError(f"{type(input_tables).__name__} is not supported")
    
    def register_derived_table(self, df:pd.DataFrame, name:str, table_description:str ):
        card = Catalog.dataframe_to_table_card( df, name, table_description, 'derived' )
        self._catalog.register_table( card )
        self.conn.register(name, df)

    def sanitize_df(self, df):

        df = df.copy()
        return df 
    
        # Ensure index is not problematic
        if df.index.name is not None or not isinstance(df.index, pd.RangeIndex):
            df = df.reset_index()

        # Attempt to convert object columns
        for col in df.columns:
            if df[col].dtype == "object":
                # try datetime
                converted = pd.to_datetime(df[col], errors="ignore")
                if not pd.api.types.is_object_dtype(converted):
                    df[col] = converted
                    continue

                # try numeric
                converted = pd.to_numeric(df[col], errors="ignore")
                if not pd.api.types.is_object_dtype(converted):
                    df[col] = converted

        return df

    def get_table_names( self ):
        return [name for name in self._catalog.tables ] 
    
    def get_tables_creation_datetime( self )-> Dict[str,str]  :
        return { t: v.creation_date  for t,v in self._catalog.tables.items() }   # pyright: ignore[reportReturnType]
    
    def get_tables_brief_description( self ):
        return { t: v.description  for t,v in self._catalog.tables.items() }  


data = SmartData()
data.initialize_from_named_dataframes( df_dict, named_table_models)
data.register_derived_table( df, "a derived table", 'Something created on the fly')

print(data.catalog_snapshot( ['injectors'] ))



tables:
- name: injectors
  description: Water-injection time series per injector well, subzone and sector
  kind: base
  creation_date: '2026-04-14 22:15:00.147880'
  row_count: 490
  columns:
  - name: DATE
    data_type: timestamp
    description: Timestamp of injection observation.
  - name: NAME
    data_type: string
    description: Injector well name. Unique identifier for the injector well
  - name: WATER_INJECTION_VOLUME
    data_type: float
    description: Injected water volume for the period.
  - name: SUBZONE
    data_type: string
    description: Subzone name. A subzone indicates vertical interval
  - name: SECTOR
    data_type: integer
    description: Sector identifier. A sector indicates a geographical location
  - name: YEAR
    data_type: integer
    description: Year component of DATE.
  - name: MONTH
    data_type: integer
    description: Month component of DATE.
  - name: DAY
    data_type: integer
    description: Day component of DATE.
  relationships:
  - tabl

In [10]:
list(catalog.tables.values())[2:3]

[TableCard(name='aaa', description='sddgf', kind='derived', creation_date='2026-04-14 21:23:51.114341', row_count=2, columns=[ColumnCard(name='name', data_type='object', description=None), ColumnCard(name='age', data_type='int64', description=None)], relationships=[], sql_examples=[])]

In [5]:
inj = pd.read_csv("../datasets/IX5I_4P/injectors.csv")
inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
inj['DAY']   = inj['DATE'].dt.day
inj['MONTH'] = inj['DATE'].dt.month
inj['YEAR']  = inj['DATE'].dt.year
display( inj.sample(8))


llm = azure_llm_if()
print( llm )

,DATE,NAME,WATER_INJECTION_VOLUME,SECTOR,ZONE,SUBZONE,WELL_TYPE,DAY,MONTH,YEAR
300,2016-06-02,I4,375.000,1,WARA,WARA1,Injector,2,6,2016
484,2023-08-02,I5,1176.190,1,WARA,WARA1,Injector,2,8,2023
401,2016-09-02,I5,408.484,1,WARA,WARA1,Injector,2,9,2016
433,2019-05-02,I5,1373.730,1,WARA,WARA1,Injector,2,5,2019
395,2016-03-02,I5,1052.000,1,WARA,WARA1,Injector,2,3,2016
466,2022-02-02,I5,639.871,1,WARA,WARA1,Injector,2,2,2022
159,2021-01-02,I2,1324.130,1,WARA,WARA1,Injector,2,1,2021
477,2023-01-02,I5,532.323,1,WARA,WARA1,Injector,2,1,2023


client=<openai.resources.chat.completions.completions.Completions object at 0x731f85557c40> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x731f7f92e5f0> root_client=<openai.lib.azure.AzureOpenAI object at 0x731f85557460> root_async_client=<openai.lib.azure.AsyncAzureOpenAI object at 0x731f84399ff0> model_name='gpt-4o' temperature=0.0 model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True azure_endpoint='https://openai-if-test.openai.azure.com/' deployment_name='gpt-4' openai_api_version='2024-12-01-preview' openai_api_type='azure'


In [ ]:

def sanitize_df(df):

    df = df.copy()

    # Ensure index is not problematic
    if df.index.name is not None or not isinstance(df.index, pd.RangeIndex):
        df = df.reset_index()

    # Attempt to convert object columns
    for col in df.columns:
        if df[col].dtype == "object":
            # try datetime
            converted = pd.to_datetime(df[col], errors="ignore")
            if not pd.api.types.is_object_dtype(converted):
                df[col] = converted
                continue

            # try numeric
            converted = pd.to_numeric(df[col], errors="ignore")
            if not pd.api.types.is_object_dtype(converted):
                df[col] = converted

    return df

def register_table(conn, name: str, df, registry: set, columns: Dict[str, Any]):
    df = sanitize_df(df)
    conn.register(name, df)


con = duckdb.connect()
con.register("injectors", sanitize_df(inj))

In [8]:

con = duckdb.connect()
con.register("injectors", sanitize_df(inj))


/tmp/ipykernel_110903/3044984730.py:14: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  converted = pd.to_datetime(df[col], errors="ignore")
/tmp/ipykernel_110903/3044984730.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(df[col], errors="ignore")
/tmp/ipykernel_110903/3044984730.py:20: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  converted = pd.to_numeric(df[col], errors="ignore")
/tmp/ipykernel_110903/3044984730.py:14: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  conve

In [9]:
from load_semantics import load_semantics, load_idiom_rules 
semantic_catalog, sql_idioms, semantic_context = load_semantics( Path("../semantics/") )
rules, idiom_context = load_idiom_rules( idiom = 'duckdb',path = Path("../semantics/idioms.json") )
 

In [10]:

query = 'fdfgdfgdg'
prompt_template = """
You are an expert SQL generator for DuckDB based on the 
following database schema and description:

# Tables:
{context_lines}   

# Rules:
- Generate ONLY the SQL instruction. 
- Do NOT use markdown code blocks (e.g., ```sql). 
- Do NOT use prefixes or explanations.
- Do NOT end the query with a semicolon ';'.
- Example: SELECT COUNT(DISTINCT well_id) AS well_count FROM injectors

ALWAYS use {idiom} compliant SQL syntax.
Examples:
{idiom_examples}

 
# Business context:
Active wells within a given timeframe: 
- producer: liquid production > 0 within the timeframe.
- injector: water injection > 0 within the timeframe.

"Current date" refers to the MAX("DATE") in the dataset.

Summarization of injection: high-level figures on current 
active injectors, the total injection volume in each of the last three months 
split by subzones. The summary must indicate the number of inactive 
injectors. 

# Task: 
{user_query}

# {idiom} SQL:
"""


idiom_name = "duckdb"
idiom_examples = idiom_context# "\n".join([f"- {k}: {v}" for k, v in idioms[idiom_name].items()])

context_dict = semantic_catalog.tables[0].model_dump(exclude_none=True)
context_yaml = yaml.dump(context_dict, sort_keys=False)


# FIX: Changed 'idioms' to 'idiom' to match the template placeholder
prompt = prompt_template.format(
    user_query=query, 
    context_lines=context_yaml, 
    idiom=idiom_name, 
    idiom_examples=idiom_examples
) 

# response = llm.invoke(prompt)
print(prompt)


You are an expert SQL generator for DuckDB based on the 
following database schema and description:

# Tables:
name: injectors
description: Water-injection time series per injector well, subzone and sector
columns:
- name: DATE
  data_type: timestamp
  description: Timestamp of injection observation.
  business_rules: []
- name: NAME
  data_type: string
  description: Injector well name. Unique identifier for the injector well
  business_rules: []
- name: WATER_INJECTION_VOLUME
  data_type: float
  description: Injected water volume for the period.
  business_rules: []
- name: SUBZONE
  data_type: string
  description: Subzone name. A subzone indicates vertical interval
  business_rules:
  - Expected values include labels such as LW, RW, UW, Unique.
- name: SECTOR
  data_type: integer
  description: Sector identifier. A sector indicates a geographical location
  business_rules:
  - Should be an integer sector id.
- name: YEAR
  data_type: integer
  description: Year component of DATE.

In [ ]:
query1 = "how many wells are there?"
query2 = "What is the total water injection volume by year?"
query3 = "Tell me the mean yearly water injection volume for each subzone"
query4 = "rank wells by their variability (std) in water injection volume (the higher the grater the rank)?"
query5 = "whats the frequency of observations in the dataset (D, M, Y) ?"
query6 = "summarize the injection data"
query7 = "Which well had the single highest WATER_INJECTION_VOLUME reading at any point in time and what was that reading?"
query8 = "What is the average monthly injection volume per well grouped by NAME and MONTH?"
query9 = """For each SUBZONE compute the year-over-year percentage change in total injection 
volume and report the largest drop
"""


queries = [
    #(query1, lambda x: int(x.loc[0,:].values[0]) == 5, 1),
    #(query2, lambda x: abs(float(x.loc[ x["YEAR"] == 2016, :].values[0][1]) - 56202.305) < 0.01, 2),
    #(query3, lambda x: abs(1306.96 - float(x.set_index(["YEAR", "SUBZONE"]).iloc[:, 0].loc[(2019, "WARA1")])) < 0.01, 2),
    #(query4, lambda x: (x.loc[x["NAME"] == "I1", x.columns[-1]] == 1).any(), 2),
    #(query7, lambda x: (x.shape[0] == 1 and (x.iloc[0]["NAME"] == "I1") and abs(float(x.iloc[0]["WATER_INJECTION_VOLUME"]) - 3537.0) < 0.1), 1),
    (query8, lambda x: False, 2),
    #(query9, lambda x: False, 3),
]

for n, query_item in enumerate(queries):
    query, checking_fn, complexity = query_item
    print(60 * "=")
    print(f"Complexity index: {complexity}")
    print(query)
   
    prompt = prompt_template.format(
        user_query=query, 
        context_lines=context_yaml, 
        idiom=idiom_name, 
        idiom_examples=idiom_examples
    )
    response = llm.invoke(prompt)

    print(response)

    # execute the generated SQL
    sql = response.content.strip()
    print(sql)

    try:
        result = con.execute(sql).fetchdf()
        display(result.sample(min(3, result.shape[0])))

        # check result
        print("success", checking_fn(result))
    except Exception as e:
        print("error", e)


Complexity index: 2
What is the average monthly injection volume per well grouped by NAME and MONTH?
content='SELECT NAME, MONTH, AVG(WATER_INJECTION_VOLUME) AS avg_monthly_injection_volume \nFROM injectors \nGROUP BY NAME, MONTH' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 813, 'total_tokens': 843, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_af7f7349a4', 'id': 'chatcmpl-DTs1lOR6pF0cBkNcFCMpUHjy7VJdw', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filter

,NAME,MONTH,avg_monthly_injection_volume
32,I1,1,1472.056125
35,I1,6,1506.976250
26,I5,2,825.250375


success False


In [28]:
from pprint import pprint 
pprint(response.usage_metadata)
pprint(response.response_metadata["token_usage"]) 

{'input_token_details': {'audio': 0, 'cache_read': 0},
 'input_tokens': 820,
 'output_token_details': {'audio': 0, 'reasoning': 0},
 'output_tokens': 237,
 'total_tokens': 1057}
{'completion_tokens': 237,
 'completion_tokens_details': {'accepted_prediction_tokens': 0,
                               'audio_tokens': 0,
                               'reasoning_tokens': 0,
                               'rejected_prediction_tokens': 0},
 'prompt_tokens': 820,
 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
 'total_tokens': 1057}


# Now lets make it agentic and add some sort of structured output 

## Generate a prompt_builder that RAGs the questions, reasoning and sql 
## Add custom tools 


In [ ]:
response.response_metadata['token_usage']['prompt_tokens']

In [ ]:


class SmartData:
    def __init__(self, llm=None):
        self.con = duckdb.connect()
        self.tables = set()
        self.semantic = {}
        self.semantic_model = load_semantic_model()
        self.columns = {}
        self._llm = llm

    # -----------------------------
    # LLM PROPERTY
    # -----------------------------
    @property
    def llm(self):
        return self._llm

    @llm.setter
    def llm(self, model):
        self._llm = model

    # -----------------------------
    # CORE EXECUTION
    # -----------------------------
    def execute_query(self, sql: str):
        return self.con.execute(sql).fetchdf()

    # -----------------------------
    # TABLE REGISTRATION
    # -----------------------------
    def register_tables(self, tables: Dict[str, Any]):
        register_tables(self.con, tables, self.tables, self.columns)

    def register_table(self, name: str, df):
        register_table(self.con, name, df, self.tables, self.columns)

    # -----------------------------
    # SEMANTIC REGISTRATION
    # -----------------------------
    def register_semantic(self, name: str, description: str):
        if name not in self.tables:
            raise ValueError(f"Table '{name}' is not registered")
        self.semantic[name] = description

    # -----------------------------
    # SQL GENERATION
    # -----------------------------
    def generate_sql(self, user_query: str, **kwargs) -> str:
        if self._llm is None:
            raise ValueError("LLM is not set")

        context_lines = []
        for table in self.tables:
            desc = self.semantic.get(table, "")
            cols = self.columns.get(table, [])
            context_lines.append(
                f"Table: {table}\nDescription: {desc}\nColumns: {', '.join(cols)}"
            )

        context = "\n\n".join(context_lines)

        prompt = f"""
You are an expert SQL generator for DuckDB.

Available tables:
{context}

User request:
{user_query}

Generate a valid DuckDB SQL query only.
"""

        return self._llm(prompt, **kwargs)


llm = azure_llm_if()
print( llm )

